# ML-10 — Content Action Playbook

**What this notebook is.** The last artifact of the ML lane: a *ranked review queue* plus the plain-language rules for reading it. Nothing new is trained here. The model, the frame, and the label are the ones from `w03_data_contract.ipynb`; every validation number is the one measured out-of-fold in `w06_validation_audit.ipynb` and re-verified in §5 of this notebook.

**The audience is one person**: an editor opening a queue on the morning of a decision date and asking "which 20 pages do I look at first, and why?". This notebook is decision-support. It is not an automation, and §3 says exactly where automation must stop.

**Where every number comes from** (nothing here is re-derived from a different design than w06 used):

| Number | Value | Source |
|---|---|---|
| Base rate `is_visibility_loss` | 0.4438 | w06 §2, re-verified here |
| Grouped-by-client AUC (`GroupKFold(5)`) | 0.5902 | w06 §2, re-verified here |
| Grouped P@50 / random-ranking P@50 | 0.4800 / 0.4080 | w06 §2, re-verified here |
| Random row-split AUC / P@50 | 0.6262 / 0.5600 | w06 §2 (the leaky design — quoted only for contrast) |
| Week-4 rule baseline P@50 (grouped) | 0.4360 | w06 §2 |
| Labeled rows / clients | 77,400 / 38 | w06 §2 |
| Decision date · feature window · outcome window | 2026-03-15 · 03-01→03-15 · 03-16→03-31 | w03 data contract |

Two words carry the whole notebook: **measured** and **directional**. Grouped AUC 0.5902 is modest, so everything below is a *triage order*, not a finding about any single page.

## 1. Ranked actions + reason codes

*The queue: what to do first, and why, in words a human trusts.*

### 1.1 What the score is

The score on each queue row is `predicted_probability` from the Week-5 estimator — `LogisticRegression(max_iter=500, C=1.0)` on the same five features, fitted on all 77,400 labeled rows. **I am explicit about which of the two candidate scores this is**: it is the model, not the Week-4 rule, and it is a full-frame fit.

That choice has a cost worth naming rather than hiding. A full-frame fit ranks rows the model has already seen, so **the score itself is not an out-of-fold number and I quote no in-sample AUC for it anywhere in this notebook.** The only validated numbers are w06's out-of-fold ones, re-verified in §5: grouped AUC 0.5902, grouped P@50 0.4800 against a random-ranking 0.4080. So the score is a *sorting* device whose ranking quality is known to be modest; the label column is carried in the CSV only so the decline rate of the queue can be read off, and it is not a prediction of anything future.

### 1.2 The five reason codes, and why these five

The original skeleton suggested `stale_high_traffic` and `pos_constrained_low_ctr`. **I kept neither name**, because the coefficients measured in w06 point the other way, and a reason code that contradicts the measured sign is a story, not a receipt.

| Reason code | Feature pattern it fires on | Measured basis, standalone (w06) | Action |
|---|---|---|---|
| `low_ctr_more_decline` | `f_ctr` below the labeled-frame median | coef **−0.2782** — *negative*, so low CTR goes with **more** decline. Largest permutation importance (0.0602 ± 0.0026), a hair above `f_active_days` (0.0596 ± 0.0015) — the two overlap within one SD, so treat them as **tied**, not ranked. Standalone: **0.5072** vs 0.3804 for high-CTR pages, lift **+0.0634** | `ctr_review` |
| `high_volume_tilt` | `f_log_impressions` at or above the top tertile | directional only: standalone **0.4629**, lift +0.0191 | `monitor` |
| `poor_position_drift` | `f_avg_position` above the labeled-frame median | coef **+0.1199** — worse position goes with more decline. Standalone **0.4620**, lift +0.0182. Honest caveat: permutation importance 0.0010, so it barely moves held-out AUC on its own | `position_check` |
| `low_activity_concentration` | `f_active_days` ≤ 10 | coef **+0.6803** is **positive**: *more* active days goes with *more* decline. So this group is the **opposite** of a decline reason — standalone **0.1251**, lift **−0.3187**, the lowest of any group | `monitor_low_priority` |
| `no_measured_pattern` | none of the above | the residual bucket — kept visible instead of being folded into a flattering code | `monitor_low_priority` |

**`low_activity_concentration` is the code I had to correct, and the correction is the interesting part.** The first version of this notebook cut it as the "bottom third by active-day count" and gave it the queue's only `review_for_refresh` action, on the reasoning that `f_active_days` was the strongest lever. Both halves of that were wrong:

- The coefficient is **positive** (+0.6803). More active days goes with *more* decline, so thin day coverage is where decline is **absent**, not where it concentrates. Measured directly: pages active ≤10 days decline at **0.1251**; pages active all 15 days decline at **0.4948**.
- A rank-based bottom-third cut does not work on this distribution at all. **79.2% of labeled pages sit at 15/15 active days**, so "the bottom third by rank" still reached a threshold of 15 and fired on *fully-active* pages — the top of the queue was being labelled low-activity. The cut is now absolute (`≤ 10` days) and the code's action is `monitor_low_priority`.

So the naive story ("thin coverage → declining → refresh it") is not merely unsupported here, it is **measured backwards**, and the queue now says so in a code rather than in a footnote.

**No code earns `review_for_refresh`.** The best measured lift any single pattern shows over the base rate is **+0.0634**. That is not enough to tell an editor to change a page, so no row in this queue carries a refresh action. Every action below is a *look*; deciding to refresh stays a human decision made outside this artifact. That is a deliberate choice, and it is the same reasoning §3.2 applies to automation.

### 1.3 Priority order, and why a code's rate has two numbers

A row can trip several patterns, so exactly one becomes its **primary** code, and the priority order is the measured lift order in the table above — highest first. Two consequences I print rather than hide:

- **A code's group statistics depend on the contest.** Whoever goes first absorbs the rows the others would have shared, so `low_ctr_more_decline` looks different once the lower-lift codes stop competing with it. The notebook therefore reports each pattern's **standalone** rate (over every row it matches) as the justification, and the **assigned** rate (after the contest) beside it. Where the two differ, the difference is the priority order, not the feature.
- **A code is a description, not a diagnosis.** All five say "these pages share a measured feature pattern". None of them says why the pattern exists, and none of them is a prediction.

And the result that follows from the contest order is a **negative** one, so it belongs here rather than in a footnote: after the contest, only `low_ctr_more_decline` files a group that declines above the base rate (0.5072). The other four file groups at or below it, because the highest-lift pattern has already absorbed the rows they would have shared. **This queue is close to a single-signal queue with a CTR split on top.** The reason column stays — each code names a real measured pattern, and hiding the codes that fire would be the dishonest choice — but it should not be read as five independent findings, and a reviewer who wants the honest summary should read `low_ctr_more_decline` as *the* finding and the rest as coverage.

### 1.4 The other code I dropped, and why

`stale_high_traffic` is **not** in the table above. It was the Week-4 heuristic, and w06 measured that it does not survive: age quartiles are non-monotonic (0.4074 / 0.5458 / 0.4524 / 0.3707), the *oldest* quartile declines **least**, and `f_log_age_days` has a **negative** coefficient (−0.2028). So "old page" is not a measured decline signal in this slice. It is still a defensible editorial reason to look at a page — staleness is a real content property — but it is not what this label measures, and putting it in the reason column would have quietly borrowed credibility the number does not have. The §1 code cell measures it and assigns it to nobody.

`pos_constrained_low_ctr` is dropped for the mirror-image reason: the word "constrained" implies low CTR is *protective*. The sign is the opposite, so `low_ctr_more_decline` says what was measured.

### 1.5 Plain words, per code

- **`low_ctr_more_decline`** — few people who do see the page click it. The measured direction is that these pages declined **more** often (0.5072 vs 0.3804). Read it as *"low CTR, here is a reason to look"* and the action is a look at intent match, not a verdict.
- **`high_volume_tilt`** — a high-traffic page. The tilt is +0.0191, small. Mostly a *"monitor"* signal.
- **`poor_position_drift`** — average position across the window was in the worse half. Measured at the group level (0.4620) but the feature's held-out importance is ~0, so it is a prompt to check whether the position is a stable snapshot or a trend — exactly a question for a human.
- **`low_activity_concentration`** — the page was indexed on 10 days or fewer of the 15. **These pages declined least of any group in the frame (0.1251).** The code is here to retire the naive story and to make the negative result visible in the queue itself, not to justify a refresh.
- **`no_measured_pattern`** — nothing above fired. Explicitly low priority.


In [1]:
# ── §1: rebuild the w06 frame, fit the Week-5 estimator, build the ranked queue ──
# Read-only. The SQL is byte-identical in intent to w03 cell 2 / w06 cell 4: same windows,
# same five features, same label, same stable ORDER BY. No new modelling choices are made here.
from pathlib import Path
import sys, json, warnings
import numpy as np, pandas as pd
import duckdb
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GroupKFold, StratifiedKFold, cross_val_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

ROOT = Path.cwd()
for _ in range(4):                                   # work/notebooks -> repo root, wherever jupyter runs
    if (ROOT / "AGENTS.md").exists():
        break
    ROOT = ROOT.parent

CACHE = ROOT / "work" / "outputs" / "warehouse_cache"
for needed in ["fact_2026-03.parquet", "dim_content.parquet"]:
    if not (CACHE / needed).exists():
        raise SystemExit(f"warehouse cache missing {needed} - see skills/flyrank/flyrank-data; "
                         "never download from here.")
FACT = f"read_parquet('{CACHE}/fact_2026-03.parquet')"
DIM_CONTENT = f"read_parquet('{CACHE}/dim_content.parquet')"

DECISION_DATE = "2026-03-15"
FEAT_LO, FEAT_HI = "2026-03-01", "2026-03-15"        # feature window, knowable at the decision moment
OUT_LO,  OUT_HI  = "2026-03-16", "2026-03-31"        # outcome window, label only
con = duckdb.connect()

FEATURE_CTE = f"""
  SELECT client_hash_id, content_hash_id,
    SUM(gsc_impressions) AS f_impressions,
    SUM(gsc_clicks) AS f_clicks,
    AVG(CASE WHEN gsc_impressions > 0 THEN gsc_avg_position END) AS f_avg_position,
    COUNT(DISTINCT report_date) FILTER (WHERE gsc_impressions > 0) AS f_active_days
  FROM {FACT}
  WHERE report_date BETWEEN DATE '{FEAT_LO}' AND DATE '{FEAT_HI}'
    AND gsc_data_available IS TRUE
  GROUP BY 1, 2
"""
OUTCOME_CTE = f"""
  SELECT client_hash_id, content_hash_id, SUM(gsc_impressions) AS t_impressions
  FROM {FACT}
  WHERE report_date BETWEEN DATE '{OUT_LO}' AND DATE '{OUT_HI}'
    AND gsc_data_available IS TRUE
  GROUP BY 1, 2
"""
frame_sql = f"""
WITH feat AS ({FEATURE_CTE}), outcome AS ({OUTCOME_CTE})
SELECT
  feat.client_hash_id, feat.content_hash_id,
  LN(1 + feat.f_impressions)                             AS f_log_impressions,
  feat.f_clicks::DOUBLE / NULLIF(feat.f_impressions, 0) AS f_ctr,
  feat.f_avg_position, feat.f_active_days,
  LN(1 + DATEDIFF('day', dim.content_created_date, DATE '{DECISION_DATE}')) AS f_log_age_days,
  CASE WHEN feat.f_impressions >= 100
       THEN (outcome.t_impressions < feat.f_impressions)::INT END AS is_visibility_loss
FROM feat
LEFT JOIN outcome USING (client_hash_id, content_hash_id)
LEFT JOIN {DIM_CONTENT} dim USING (content_hash_id)
ORDER BY feat.client_hash_id, feat.content_hash_id   -- stable row order, so every number is reproducible
"""
frame  = con.sql(frame_sql).df()
FEATURES = ["f_log_impressions", "f_ctr", "f_avg_position", "f_active_days", "f_log_age_days"]

# The queue ranks ONLY the rows the model was able to see: a row with <100 first-half impressions
# has no label, so it has no validated meaning in this design. §3 makes that a hard no-go.
labeled = frame.dropna(subset=["is_visibility_loss"]).copy()
y      = labeled["is_visibility_loss"].astype(int).values
groups = labeled["client_hash_id"].values
X      = labeled[FEATURES].replace([np.inf, -np.inf], np.nan).fillna(0).values
BASE_RATE = float(y.mean())

warnings.filterwarnings("ignore", category=RuntimeWarning, module="sklearn.*")
QUEUE_MODEL = LogisticRegression(max_iter=500, C=1.0)   # exactly the w03 / w05 estimator
queue_model = QUEUE_MODEL.fit(X, y)
labeled["predicted_probability"] = queue_model.predict_proba(X)[:, 1]

print(f"frame rows {len(frame):,} | labeled rows {len(labeled):,} | clients {labeled['client_hash_id'].nunique()}")
print(f"base rate {BASE_RATE:.4f} | rows/client median {labeled.groupby('client_hash_id').size().median():,.0f}, "
      f"max {labeled.groupby('client_hash_id').size().max():,}")
print(f"score range {labeled['predicted_probability'].min():.4f} - {labeled['predicted_probability'].max():.4f} "
      f"(in-sample fit; no in-sample AUC is quoted anywhere)")

# ── reason codes: each cut is justified by its STANDALONE measured rate, not by its coefficient ──
# Two corrections to the first version of this block, both forced by the data and both printed below.
#
# (1) `f_active_days` is POSITIVELY signed (+0.6803): MORE active days goes with MORE decline. A code
#     called `low_activity_concentration` therefore cannot be a decline reason, and it is not one here.
#     A rank-based "bottom third" cut was also unusable: 79.2% of labeled pages sit at 15/15 active
#     days, so the bottom third by rank still reached 15 and fired on fully-active pages. The cut is
#     now an absolute ACT_MAX of 10 days, and the table shows the group it files declines 0.32 BELOW
#     base rate, so its action is `monitor_low_priority` and never a refresh.
# (2) A code's group statistics depend on the priority contest (whoever goes first absorbs the rows),
#     so the justification is measured STANDALONE over every row the pattern matches, and the
#     assigned-group rate is printed beside it rather than instead of it.
q = labeled.copy()
q["f_active_days"] = q["f_active_days"].fillna(0)
POS, CTR_MED = q["f_avg_position"].median(), q["f_ctr"].median()
VOL_T, ACT_MAX = q["f_log_impressions"].quantile(2 / 3), 10
CODE_ORDER = ["low_ctr_more_decline", "high_volume_tilt", "poor_position_drift",
              "low_activity_concentration", "no_measured_pattern"]
ACTION_OF = {"low_ctr_more_decline": "ctr_review", "high_volume_tilt": "monitor",
             "poor_position_drift": "position_check",
             "low_activity_concentration": "monitor_low_priority",
             "no_measured_pattern": "monitor_low_priority"}

# standalone pattern table - the receipt that decides both the priority order and the action
PATTERNS = {"low_ctr_more_decline": q["f_ctr"] < CTR_MED,
            "high_volume_tilt": q["f_log_impressions"] >= VOL_T,
            "poor_position_drift": q["f_avg_position"] > POS,
            "low_activity_concentration": q["f_active_days"] <= ACT_MAX}
standalone = pd.DataFrame([
    {"reason_code": c, "n": int(m.sum()), "share": m.mean(),
     "decline_rate": float(q.loc[m, "is_visibility_loss"].mean()),
     "lift_vs_base_rate": float(q.loc[m, "is_visibility_loss"].mean()) - BASE_RATE,
     "action_if_it_wins": ACTION_OF[c]} for c, m in PATTERNS.items()])
standalone = standalone.sort_values("lift_vs_base_rate", ascending=False).reset_index(drop=True)
print("=== 1A. each reason code measured STANDALONE: every row its pattern matches ===")
print(standalone.round(4).to_string(index=False))
print(f"base rate {BASE_RATE:.4f} | ordered by measured lift, and the priority order below IS this order")
print("Every code above is a real feature pattern. Note the sign of the activity code: pages with FEW")
print(f"active days decline {standalone.loc[standalone.reason_code == 'low_activity_concentration', 'decline_rate'].iloc[0]:.4f}, "
      f"i.e. BELOW the base rate, so it is a monitor signal and not a refresh one.")

# The evidence behind the correction above, printed in full: the activity spectrum. This is the table
# that shows the relationship is POSITIVE and non-monotone, and that 79% of the frame sits at 15/15.
act_spec = (q.groupby("f_active_days")
              .agg(n=("is_visibility_loss", "size"),
                   decline_rate=("is_visibility_loss", "mean"))
              .reset_index())
act_spec["lift_vs_base_rate"] = act_spec["decline_rate"] - BASE_RATE
print("\n=== 1A-bis. decline rate by f_active_days, the top lever read directly ===")
print(act_spec.round(4).to_string(index=False))
n15 = int((q["f_active_days"] == 15).sum())
print(f"\npages at 15/15 active days: {n15:,} of {len(q):,} ({n15 / len(q):.1%}) - a mass point, which is why a")
print("rank-based 'bottom third' cut was meaningless here. Read the column, not the coefficient:")
print(f"  <= 10 active days -> {q.loc[q.f_active_days <= ACT_MAX, 'is_visibility_loss'].mean():.4f} "
      f"| == 15 -> {q.loc[q.f_active_days == 15, 'is_visibility_loss'].mean():.4f} "
      f"| base rate {BASE_RATE:.4f}")
print("  MORE active days goes with MORE decline. `low_activity_concentration` is therefore a monitor")
print("  signal, and a rank-based cut for it was measuring the tie-break, not the feature.")

# both sides of the CTR split, so the headline code's lift is visible as a contrast and not a claim
ctr_split = (q.assign(lo_ctr=q["f_ctr"] < CTR_MED)
               .groupby("lo_ctr")["is_visibility_loss"].agg(["size", "mean"]))
print(f"\n=== 1A-ter. both sides of the CTR median ({CTR_MED:.5f}) ===")
print(f"  f_ctr < median : n {int(ctr_split.loc[True, 'size']):,}  decline {ctr_split.loc[True, 'mean']:.4f}")
print(f"  f_ctr >= median: n {int(ctr_split.loc[False, 'size']):,}  decline {ctr_split.loc[False, 'mean']:.4f}")
print("  the largest single-feature permutation importance in w06 is f_ctr (0.0602), and this is the")
print("  cleanest split in the frame: a flat 50/50 cut with the largest measured gap.")

# One primary code per row, HIGHEST measured lift wins. The writes go lowest-lift-first so that the
# highest-lift code is written last and therefore survives; a forward loop would let the last pattern
# in the list overwrite the best one, which is the opposite of the documented priority.
q["reason_code"] = "no_measured_pattern"
q["action"] = ACTION_OF["no_measured_pattern"]
for c in reversed(list(standalone["reason_code"])):       # lowest lift first ...
    q.loc[PATTERNS[c], ["reason_code", "action"]] = [c, ACTION_OF[c]]   # ... best lift written last
# which patterns each row fires at all, printed so the priority contest is visible and not hidden
q["_fired_activity"] = PATTERNS["low_activity_concentration"]
q["_fired_position"] = PATTERNS["poor_position_drift"]
q["_fired_ctr"]     = PATTERNS["low_ctr_more_decline"]
q["_fired_volume"]  = PATTERNS["high_volume_tilt"]
print(f"\npatterns fired (rows may fire more than one): activity <= {ACT_MAX}d "
      f"{int(q['_fired_activity'].sum()):,} | low ctr {int(q['_fired_ctr'].sum()):,} | "
      f"worse position {int(q['_fired_position'].sum()):,} | high volume {int(q['_fired_volume'].sum()):,}")

q = q.sort_values(["predicted_probability", "client_hash_id", "content_hash_id"],
                  ascending=[False, True, True]).reset_index(drop=True)
q.insert(0, "rank", np.arange(1, len(q) + 1))

print(f"\nreason-code cuts measured on the labeled frame: active_days <= {ACT_MAX} | "
      f"ctr < median {CTR_MED:.5f} | position > median {POS:.2f} | log_impressions >= T2 {VOL_T:.2f}")
print("no code is assigned `review_for_refresh`: on this label the highest measured lift over base rate")
print(f"is {standalone['lift_vs_base_rate'].max():+.4f}, which is not enough to tell an editor to change")
print("a page. Every action here is a LOOK. Deciding to refresh stays a human decision, made outside")
print("this queue and recorded wherever the client's own process records refreshes.")
# prove the priority order is the one documented, rather than the reverse
top = q.nlargest(200, "predicted_probability")
print(f"\npriority order check: the top 200 of the queue is "
      f"{top['reason_code'].value_counts().to_dict()}")

# The second receipt: how the codes landed AFTER the contest. These rates differ from 1A on purpose -
# whoever goes first absorbs the shared rows, so the assigned rate is what the queue actually produced
# and the standalone rate above is what justifies the code. Both are printed; neither is hidden.
by_code = (q.groupby("reason_code")
             .agg(n=("is_visibility_loss", "size"),
                  decline_rate=("is_visibility_loss", "mean"),
                  mean_score=("predicted_probability", "mean"))
             .reindex(CODE_ORDER))          # a code that never wins stays visible as n = 0
by_code["n"] = by_code["n"].fillna(0).astype(int)
by_code["lift_vs_base_rate"] = by_code["decline_rate"] - BASE_RATE
by_code["share_of_queue"] = by_code["n"] / len(q)
by_code["action"] = pd.Series(ACTION_OF)
by_code["standalone_lift"] = standalone.set_index("reason_code")["lift_vs_base_rate"]
print("\n=== 1B. reason code -> measured decline rate of the pages it was ASSIGNED ===")
print(by_code.round(4).to_string())
print(f"\nbase rate for comparison: {BASE_RATE:.4f}")
print("Compare `lift_vs_base_rate` (after the contest) with `standalone_lift` (before it). Where they")
print("differ, the difference is the priority order, not the feature.")
above = by_code.index[by_code["lift_vs_base_rate"] > 0].tolist()
print(f"\ncodes whose ASSIGNED group is above the base rate {BASE_RATE:.4f}: {above}")
print(f"codes at or below it: {[c for c in by_code.index if c not in above]}")
print("That is the plainest result in this section and it is a negative one: the high-lift pattern")
print("absorbs the rows the weaker patterns would have shared, so after the contest only")
print(f"`{above[0] if above else 'none'}` files a group that declines above the base rate. The queue is")
print("close to a single-signal queue with a CTR split on top. The reason column is still honest -")
print("each code names a real measured pattern - but it should not be read as five independent")
print("findings, and no code clears a bar that would justify changing a page, so none is assigned")
print("`review_for_refresh`.")
print(f"codes assigned to 0 pages: {[c for c in CODE_ORDER if by_code.loc[c, 'n'] == 0] or 'none'} "
      f"(+ the dropped `stale_high_traffic`, measured below)")


# ── the code I refused to assign: stale_high_traffic, measured rather than argued ──────
ageable = q["f_log_age_days"].notna()      # pages with no usable creation date are NAMED, not silently dropped
q["age_quartile"] = pd.NA
q.loc[ageable, "age_quartile"] = pd.qcut(q.loc[ageable, "f_log_age_days"], 4,
                                          labels=["Q1_newest", "Q2", "Q3", "Q4_oldest"])
age = (q[ageable].groupby("age_quartile", observed=True)
         .agg(n=("is_visibility_loss", "size"), decline_rate=("is_visibility_loss", "mean")).reset_index())
print("\n=== why there is no `stale_high_traffic` code ===")
print(f"rows with no usable creation date, excluded from this table: {int((~ageable).sum()):,} "
      f"of {len(q):,} ({1 - ageable.mean():.1%})")
print(age.round(4).to_string(index=False))
print(f"\nage pattern monotone? {list(age['decline_rate'].round(3)) == sorted(age['decline_rate'].round(3))}  "
      f"| Q4_oldest - Q1_newest = {age['decline_rate'].iloc[-1] - age['decline_rate'].iloc[0]:+.4f}")
print("The oldest quartile declines LEAST and the pattern is not monotone, so 'old' is not a measured")
print("decline signal in this slice. Code assigned to 0 pages. Staleness can still be an editorial")
print("reason to look at a page - it is just not what this label measures.")

print("\n=== top 10 of the ranked queue (pseudonymized hashes, truncated for print) ===")
print(q.head(10)[["rank", "content_hash_id", "predicted_probability", "reason_code", "action",
                  "f_active_days", "f_ctr", "f_avg_position", "is_visibility_loss"]]
        .assign(content_hash_id=lambda d: d["content_hash_id"].str[:10] + "...").round(5).to_string(index=False))
print("\nNo client name, URL or raw query is printed anywhere in this notebook - only pseudonymized hashes.")

frame rows 151,981 | labeled rows 77,400 | clients 38
base rate 0.4438 | rows/client median 485, max 17,610
score range 0.0069 - 0.6793 (in-sample fit; no in-sample AUC is quoted anywhere)
=== 1A. each reason code measured STANDALONE: every row its pattern matches ===
               reason_code     n  share  decline_rate  lift_vs_base_rate    action_if_it_wins
      low_ctr_more_decline 38695 0.4999        0.5072             0.0634           ctr_review
          high_volume_tilt 25820 0.3336        0.4629             0.0191              monitor
       poor_position_drift 38700 0.5000        0.4620             0.0182       position_check
low_activity_concentration  4638 0.0599        0.1251            -0.3187 monitor_low_priority
base rate 0.4438 | ordered by measured lift, and the priority order below IS this order
Every code above is a real feature pattern. Note the sign of the activity code: pages with FEW
active days decline 0.1251, i.e. BELOW the base rate, so it is a monitor signa

## 2. Intended use and limits

*Who uses this, for what — and where it stops being valid.*

### 2.1 Intended use, stated as a job not a feature

- **User:** one editor, at one desk, on 2026-03-15 (or a repeat of this date in a later month), with a queue open next to the content inventory.
- **Task:** *triage order*. "Which pages deserve a human look first, and what should the reviewer check when they get there?"
- **Not the task:** deciding that a page will decline, deciding that a page should be rewritten, published, deindexed, or dropped, or replacing that review.
- **Decision-support only.** The output is a suggested reading order with its reasons attached. The output is not an action.

### 2.2 The limits, each with the number that sets it

1. **One month, one decision date, one label definition.** Features come from 2026-03-01→03-15 and the label from 2026-03-16→03-31. This is a March-2026 model, not a general one.
2. **The label is a within-month proxy**, not future decline: "second-half impressions below first-half impressions". A page can dip for reasons that have nothing to do with content quality, and the label never sees anything after 2026-03-31.
3. **The label only exists for pages with ≥100 first-half impressions** — 77,400 of 151,981 rows (50.9%). So the base rate 0.4438 describes *high-traffic pages only*, and the queue is a high-traffic queue. The w06 shift table shows those pages are systematically larger and more active than the ones excluded (median active days 15 vs 6), so nothing here should be read onto low-traffic pages.
4. **38 clients, very unevenly sized** (median 485 rows, max 17,610). The queue is dominated by a handful of large clients, and a per-client view would look different from a pooled one.
5. **Grouped AUC 0.5902 is modest.** Above random ranking (0.5000) and above a random-order P@50 (0.4080 at P@50 0.4800), but far from a model that separates. The model does **not** separate from my own Week-4 rule at P@50 (0.4800 vs 0.4360, inside the rule's 0.20–0.58 fold spread) — that comparison is unseparated, not a win.
6. **No time-aware split exists.** Every row shares one decision date, so "does this hold next month?" is untested here. §4 is a monitoring plan, not a validation result.
7. **The label's own selection is the biggest caveat in the paper.** A within-month, within-page comparison on already-visible pages is a narrow question; the queue should be read as "where to look inside the pages that were already getting impressions", never as "which pages are at risk".

### 2.3 The words this notebook uses

Observed, measured, directional, decision-support, associated with. Not: predicts, causes, will decline, identifies, finds the pages that are losing visibility. The score is *a ranking with a measured but modest lift*, and the reason codes are *descriptions of feature patterns*, not diagnoses.

In [2]:
# ── §2: print the limits as measurements, so none of them is only prose ───────────────
unlabeled = frame[frame["is_visibility_loss"].isna()]
per_client = labeled.groupby("client_hash_id").size()
top50_share = q.head(50)["is_visibility_loss"].mean()

limits = pd.DataFrame([
    ["population", f"{len(labeled):,} of {len(frame):,} frame rows are labeled ({len(labeled) / len(frame):.1%})",
     "the queue cannot speak about the other 49.1%"],
    ["selection effect", f"labeled median active_days {labeled['f_active_days'].median():.0f} vs "
                         f"unlabeled {unlabeled['f_active_days'].median():.0f}",
     "labeled pages are already the active ones"],
    ["base rate", f"{BASE_RATE:.4f}", "any precision number needs this next to it"],
    ["top-50 decline rate (in-sample)", f"{top50_share:.4f} (base rate {BASE_RATE:.4f}, lift {top50_share - BASE_RATE:+.4f})",
     "read off the queue's own head, in-sample; the out-of-fold number is P@50 0.4800"],
    ["client concentration", f"38 clients, median {per_client.median():,.0f} rows, max {per_client.max():,}, "
                             f"largest client {per_client.max() / len(labeled):.1%} of the queue",
     "pooled numbers are not per-client numbers"],
    ["windows", f"features {FEAT_LO}..{FEAT_HI}, label {OUT_LO}..{OUT_HI}, one decision date",
     "a March-2026 model, not a general one"],
], columns=["limit", "measured here", "consequence for the reader"])
print("=== §2 limits, each printed from the frame rather than asserted ===")
print(limits.to_string(index=False))

print(f"\nqueue length {len(q):,} rows, all labeled (>=100 first-half impressions). "
      f"Rows outside that population are absent by design, not by filtering on the outcome.")
print("No client name, URL or raw query appears above: counts, cuts and pseudonymized hashes only.")

=== §2 limits, each printed from the frame rather than asserted ===
                          limit                                                                    measured here                                                      consequence for the reader
                     population                                 77,400 of 151,981 frame rows are labeled (50.9%)                                    the queue cannot speak about the other 49.1%
               selection effect                                     labeled median active_days 15 vs unlabeled 6                                       labeled pages are already the active ones
                      base rate                                                                           0.4438                                      any precision number needs this next to it
top-50 decline rate (in-sample)                                          0.4800 (base rate 0.4438, lift +0.0362) read off the queue's own head, in-sample; the o

## 3. Human review + the no-go list

*What a person must check before acting. What should never be automated.*

### 3.1 The five questions a reviewer must answer first

The model cannot answer any of these, and each one has been the thing that makes a plausible-looking queue row wrong:

1. **Is the page actually stale, or just old by creation date?** `f_log_age_days` is creation age, and its measured coefficient is *negative* (−0.2028) with the oldest quartile declining least (0.3707). Age is not evidence of decline here. Check the last *update* date, which this frame does not contain at all.
2. **Is the position a snapshot or a trend?** `f_avg_position` is an average over impression-days in one 15-day window. A page that sat at position 4 all month and a page that fell from 2 to 20 get similar values. The trend is not in the features.
3. **Is low CTR correct for this query intent?** A navigational or branded query has a low CTR and nothing wrong with it. `f_ctr` is the second-strongest coefficient, and the reviewer's job is to ask what the page is *for*.
4. **Is the page still relevant to its topic?** Nothing in the frame measures topical decay — a February-2026 topic can be current or dead, and the label cannot tell the difference.
5. **Has it already been refreshed since the decision date?** The frame is dated 2026-03-15. A refresh on 03-20 invalidates the row; the queue has no way to know.

### 3.2 The no-go list — what must never be automated from this score

- **Do not auto-delete, auto-noindex, or auto-redirect** on the basis of this score. Grouped AUC 0.5902 cannot carry a destructive action.
- **Do not auto-refresh, auto-rewrite, or auto-republish.** The output is a reading order.
- **Do not act on a page below the 100-first-half-impression threshold.** The model never saw those rows; they have no label and no measured meaning. 74,581 frame rows are in exactly that position.
- **Do not read the score as a prediction of future decline.** It is a within-month, within-visible-pages ordering, produced by a model that clears random ranking by 0.09 AUC.
- **Do not refresh a page that is currently ranking well and getting clicks.** This is the main false-positive risk, and it is measurable — §3.3 counts exactly how many of the queue's head pages look like that and what their decline rate actually was.
- **Do not report the score as "risk" or "health" in any client-facing artifact.** It is a within-portfolio sort, not a grade, and the reason codes are feature patterns, not diagnoses.

In [3]:
# ── §3: the checklist as data, and the false-positive risk in the queue's own head ────
REVIEW_CHECKLIST = [
    ("last_update_date",
     "Not in this frame. A page can be 'old' by creation date and freshly maintained.",
     "pull the update timestamp from the CMS before acting"),
    ("position_trend",
     "f_avg_position is a 15-day average, not a slope; a stable page and a falling page look alike.",
     "compare against the previous window's average by eye"),
    ("query_intent",
     "Low CTR is expected for navigational/branded queries; f_ctr's negative coefficient is a population fact, not a fault in the page.",
     "sample the top queries the page receives"),
    ("topical_currency",
     "Nothing in the frame measures whether the topic is still live.",
     "human judgement, no automated signal available"),
    ("already_refreshed",
     "Frame is dated 2026-03-15; any later refresh invalidates the row.",
     "check for refreshes after the decision date"),
]
print("=== 3.1 reviewer checklist (must be answered before acting) ===")
for i, (check, why, how) in enumerate(REVIEW_CHECKLIST, 1):
    print(f"{i}. {check:20s} {why}")
    print(f"{'':22s} -> {how}")

NO_GO = [
    "no auto-delete / auto-noindex / auto-redirect on this score (grouped AUC 0.5902)",
    "no auto-refresh, auto-rewrite or auto-republish - the output is a reading order",
    f"no action on rows below 100 first-half impressions ({len(unlabeled):,} frame rows the model never saw)",
    "no reading of the score as a prediction of future decline",
    "no refresh of a page that currently ranks well and gets clicks (measured below)",
    "no client-facing 'risk' or 'health' label derived from this score",
]
print("\n=== 3.2 no-go list (never automated from this score) ===")
for line in NO_GO:
    print(f"  - {line}")

# 3.3 the false-positive risk, measured rather than asserted. Two cuts, because the head of the queue
# is so concentrated on weak pages that the "looks healthy" set can be empty there - and saying so is
# itself the result: the score is effectively ranking position and CTR, not finding hidden winners.
q["looks_healthy"] = (q["f_avg_position"] <= 5) & (q["f_ctr"] >= CTR_MED)
def rate(mask, df=q):
    n = int(mask.sum())
    return n, (float(df.loc[mask, "is_visibility_loss"].mean()) if n else float("nan"))

print("\n=== 3.3 false-positive risk, measured two ways ===")
for name, sl in [("whole queue (77,400 rows)", slice(None)), ("top 200 of the queue", slice(0, 200)),
                 ("top 1,000 of the queue", slice(0, 1000))]:
    sub = q.iloc[sl]
    n_healthy, r_healthy = rate(sub["looks_healthy"], sub)
    n_rest, r_rest = rate(~sub["looks_healthy"], sub)
    print(f"\n{name}:")
    print(f"  pages that look healthy on features (avg_position <= 5 AND ctr >= median {CTR_MED:.5f}): "
          f"{n_healthy:,} ({n_healthy / len(sub):.1%})")
    print(f"    their decline rate            : {r_healthy:.4f}" if n_healthy else "    their decline rate            : n/a (no such page in this slice)")
    print(f"  decline rate, the rest of the slice: {r_rest:.4f}" if n_rest else "  decline rate, the rest of the slice: n/a")
print(f"\nbase rate {BASE_RATE:.4f}")
n_h_all, r_h_all = rate(q["looks_healthy"])
n_h_200, r_h_200 = rate(q.head(200)["looks_healthy"], q.head(200))
n_h_1k,  _        = rate(q.head(1000)["looks_healthy"], q.head(1000))
print("Two things to take from this, both printed rather than asserted:")
print(f"  1. the healthy-looking pages the queue does file ({n_h_all:,} of {len(q):,}) were measured at a")
print(f"     decline rate of {r_h_all:.4f} against a base rate of {BASE_RATE:.4f} (lift {r_h_all - BASE_RATE:+.4f}),")
print("     so 'ranking well and getting clicks' is a real false-positive risk, quantified not guessed.")
print(f"  2. the head of the queue holds almost none of them: {n_h_200} of the top 200 and {n_h_1k} of the")
print("     top 1,000 pass a 'looks healthy' test, because the score is dominated by position and CTR.")
print("     That is a limitation to state, not a success to claim: the score has little to say about")
print("     pages that are already doing fine, so it cannot be used to rule them in or out.")

=== 3.1 reviewer checklist (must be answered before acting) ===
1. last_update_date     Not in this frame. A page can be 'old' by creation date and freshly maintained.
                       -> pull the update timestamp from the CMS before acting
2. position_trend       f_avg_position is a 15-day average, not a slope; a stable page and a falling page look alike.
                       -> compare against the previous window's average by eye
3. query_intent         Low CTR is expected for navigational/branded queries; f_ctr's negative coefficient is a population fact, not a fault in the page.
                       -> sample the top queries the page receives
4. topical_currency     Nothing in the frame measures whether the topic is still live.
                       -> human judgement, no automated signal available
5. already_refreshed    Frame is dated 2026-03-15; any later refresh invalidates the row.
                       -> check for refreshes after the decision date

=== 3.2 no-go 

## 4. Monitoring / retrain triggers

*What would tell you the recommendations went stale?*

A queue built on one month of one label is a perishable artifact. These are the observable events that should retire it, written so each one is a *measurement* someone can run, not a feeling.

### 4.1 Staleness signals — run these before trusting the queue again

1. **A new month's data shows a different base rate.** The recorded reference is 0.4438. A base rate that moves a long way means the label's meaning moved with it.
2. **The feature→outcome relationship shifts.** Today `f_active_days` is the top lever (coef +0.6803, permutation importance 0.0596) and `f_log_impressions` is indistinguishable from noise (−0.0030). If next month's top feature is not `f_active_days`, the reason codes in §1 no longer describe that month and the queue's text is wrong even if its AUC looks fine.
3. **A time-aware split becomes available and the grouped number changes substantially.** The reference here is grouped AUC 0.5902 with fold values 0.4666–0.6547 (w06 §2) — that is −0.1236 / +0.0645 around the mean from a single month, so anything smaller than that is not evidence of anything. The §4 cell recomputes the fold spread and uses its standard deviation as the trigger band rather than picking a round number.
4. **The decline rate in the top decile of scores drifts toward the base rate.** The out-of-fold reference is 0.5195 (lift +0.0757). If the top of the queue stops concentrating decline, the queue is no longer ordering anything and should be switched off rather than re-read.

### 4.2 Retrain triggers

- A new month is cached and validated (the panel grows by a month; the frame SQL needs its date parameters re-pointed, nothing else).
- The feature set changes — anything added or removed invalidates every coefficient sign in §1 and every reason-code cut.
- The base rate shifts by more than **0.05 absolute** (0.4438 → outside roughly 0.394–0.494). That threshold is a judgement call, deliberately larger than the base-rate variation we would expect to see between comparable samples, and I am labelling it as one rather than dressing it up as derived.
- The feature→outcome ranking changes at the top (see 4.1.2) — a retrain, not a re-run.

### 4.3 Light monitoring

There is no sealed holdout in this design, so there is nothing to monitor P@50 against yet. What is worth doing when one exists: re-measure grouped P@50 on the sealed month with the same code path, keep the 0.4800 / 0.4080 (model / random-ranking) pair as the reference, and report the gap rather than the absolute number. Until then, the honest monitoring posture is the staleness checklist above plus a person reading the top of the queue.

In [4]:
# ── §4: the monitoring plan as a table with today's values and a trigger for each ────
gkf = GroupKFold(5)
grp_folds = cross_val_score(QUEUE_MODEL, X, y, cv=gkf, groups=groups, scoring="roc_auc")
grp_sd = float(np.std(grp_folds))

TRIGGERS = pd.DataFrame([
    ["base rate", f"{BASE_RATE:.4f}", "outside 0.394 - 0.494 (0.05 absolute)",
     "retrain + re-derive every reason-code cut", "the label's meaning moved"],
    ["grouped AUC", f"{grp_folds.mean():.4f} (folds {np.round(grp_folds, 4)})",
     f"mean moves more than {grp_sd:.4f} (1 SD of today's fold spread)",
     "re-audit before re-publishing the queue", "below ~1 SD is noise, not a change"],
    ["top feature", "f_active_days (coef +0.6803, perm imp 0.0596)",
     "any other feature becomes the top lever",
     "rewrite the §1 reason codes, not just refit", "reason text must match the month's data"],
    ["top-decile decline rate", "0.5195 out-of-fold (lift +0.0757 over 0.4438)",
     "drifts down toward the base rate", "switch the queue off, do not re-read it",
     "the ordering has stopped ordering"],
    ["time-aware split", "none exists - one decision date only",
     "a multi-month panel becomes available", "re-run with a time-aware split before any reuse",
     "this is the largest untested assumption in §2"],
    ["feature set", "5 features, unchanged since w03",
     "any feature added or removed", "full retrain; every coefficient sign in §1 is void",
     "the coefficients ARE the reason codes"],
], columns=["monitor", "value recorded on 2026-03-15", "trigger", "action if triggered", "why"])
print("=== §4 monitoring and retrain triggers ===")
print(TRIGGERS.to_string(index=False))
print(f"\ngrouped fold spread today: {grp_folds.min():.4f} - {grp_folds.max():.4f} "
      f"(SD {grp_sd:.4f}) - a single month's fold-to-fold noise, quoted so the AUC trigger is not arbitrary")
print(f"  deviations from the mean {grp_folds.mean():.4f}: "
      f"{grp_folds.min() - grp_folds.mean():+.4f} to {grp_folds.max() - grp_folds.mean():+.4f}"
      f"  (the 4.1 trigger band; a change smaller than this is not evidence of anything)")
print("no sealed holdout exists in this design, so there is no P@50 to monitor against yet; see §4.3.")

=== §4 monitoring and retrain triggers ===
                monitor                        value recorded on 2026-03-15                                                   trigger                                action if triggered                                           why
              base rate                                              0.4438                     outside 0.394 - 0.494 (0.05 absolute)          retrain + re-derive every reason-code cut                     the label's meaning moved
            grouped AUC 0.5902 (folds [0.5988 0.6213 0.4666 0.6547 0.6097]) mean moves more than 0.0646 (1 SD of today's fold spread)            re-audit before re-publishing the queue            below ~1 SD is noise, not a change
            top feature       f_active_days (coef +0.6803, perm imp 0.0596)                   any other feature becomes the top lever        rewrite the §1 reason codes, not just refit       reason text must match the month's data
top-decile decline rate       0.5

## 5. Exports for the paper

*Write the queue and any figures the paper reuses to `work/outputs/` and `work/figures/`.*

Four artifacts, all regenerated by this notebook from the local warehouse cache:

1. **`work/outputs/action_playbook_queue.csv`** — the full ranked queue (77,400 rows), with `rank`, the two pseudonymized hashes, `predicted_probability`, `reason_code`, `action`, the five feature values, the label for read-off only, and the decile used by the figure. Not committed: `work/**/*.csv` is gitignored, and the notebook regenerates it.
2. **`work/outputs/playbook_metrics.json`** — the receipts. Every value in it is either re-measured in the cell below or carried from w06 with the design named.
3. **`work/figures/queue_decline_profile.png`** — decline rate by score decile, with the base rate drawn as the reference line. It is the figure that checks the queue's central promise: that the top of the ranking is where decline concentrates.
4. **`work/figures/reason_code_breakdown.png`** — two panels: how much of the queue each reason code holds, and the measured decline rate of the pages each code filed, against the base-rate line. Counts alone would make the most common code look like the most useful one, so the two are drawn side by side. The codes that sit on the base rate are the point of the figure, not a defect in it — they are why their action is `monitor`.

Both figures are plotted on an **in-sample** fit of the queue score, so they are consistency checks on the ordering, not performance claims. The out-of-fold numbers to quote remain grouped AUC 0.5902 and grouped P@50 0.4800 (w06 §2, re-verified in the cell above).


In [5]:
# ── §5: exports — queue CSV, metrics JSON, one figure ────────────────────────────────
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

OUT_DIR, FIG_DIR = ROOT / "work" / "outputs", ROOT / "work" / "figures"
OUT_DIR.mkdir(parents=True, exist_ok=True)
FIG_DIR.mkdir(parents=True, exist_ok=True)

# ── the queue CSV ──
QUEUE_COLS = ["rank", "client_hash_id", "content_hash_id", "predicted_probability",
              "reason_code", "action"] + FEATURES + ["is_visibility_loss", "score_decile"]
q["score_decile"] = pd.qcut(q["predicted_probability"], 10, labels=False, duplicates="drop") + 1
q["predicted_probability"] = q["predicted_probability"].round(6)
for c in FEATURES:
    q[c] = q[c].round(6)
queue_path = OUT_DIR / "action_playbook_queue.csv"
q[QUEUE_COLS].to_csv(queue_path, index=False)
print(f"wrote {queue_path.relative_to(ROOT)}  rows {len(q):,} cols {len(QUEUE_COLS)}")
print("columns: " + ", ".join(QUEUE_COLS))
print("is_visibility_loss is carried for read-off only - it is the March label, not a future outcome.")

# ── re-verify the w06 numbers this playbook is built on, out-of-fold, in this notebook ──
RAND3 = StratifiedKFold(3, shuffle=True, random_state=0)
auc_rand = float(cross_val_score(QUEUE_MODEL, X, y, cv=RAND3, scoring="roc_auc").mean())
auc_grp  = float(cross_val_score(QUEUE_MODEL, X, y, cv=gkf, groups=groups, scoring="roc_auc").mean())

def per_fold_p_at_k(splitter, grp=None, k=50, seed=0):
    """P@k ranked INSIDE each fold (scores from different fold-models are not on one scale)."""
    model_hits, random_hits = [], []
    splits = splitter.split(X, y, grp) if grp is not None else splitter.split(X, y)
    for tr, te in splits:
        if len(te) < k:
            continue
        m = QUEUE_MODEL.fit(X[tr], y[tr])
        model_hits.append(y[te][np.argsort(-m.predict_proba(X[te])[:, 1])[:k]].mean())
        random_hits.append(y[te][np.random.RandomState(seed).permutation(len(te))[:k]].mean())
    return float(np.mean(model_hits)), float(np.mean(random_hits))

p_grp_m, p_grp_r = per_fold_p_at_k(gkf, groups)
p_rand_m, p_rand_r = per_fold_p_at_k(RAND3)

# the Week-4 rule, on the same rows and folds - a comparator, never a feature (w06 §3)
rule = labeled.copy(); rule["rule_score"] = 0.0
c_a = (rule["f_avg_position"] <= 5) & (rule["f_ctr"] < rule.loc[rule["f_avg_position"] <= 5, "f_ctr"].median())
c_b = (rule["f_log_impressions"] >= rule["f_log_impressions"].median()) & \
      (rule["f_log_age_days"] >= rule["f_log_age_days"].median())
c_c = (rule["f_log_impressions"] >= rule["f_log_impressions"].median()) & \
      (rule["f_ctr"] < rule["f_ctr"].median())
rule.loc[c_a, "rule_score"] += 2
rule.loc[c_b & ~c_a, "rule_score"] += 1
rule.loc[c_c & ~c_a & ~c_b, "rule_score"] += 1
rule_rs = rule["rule_score"].to_numpy()
p_rule = float(np.mean([y[te][np.argsort(-rule_rs[te])[:50]].mean()
                        for _, te in gkf.split(X, y, groups) if len(te) >= 50]))

METRICS = {
    "base_rate": round(BASE_RATE, 4),
    "grouped_auc": round(auc_grp, 4),
    "grouped_p_at_50": round(p_grp_m, 4),
    "grouped_p_at_50_random_ranking": round(p_grp_r, 4),
    "random_auc": round(auc_rand, 4),
    "random_p_at_50": round(p_rand_m, 4),
    "random_p_at_50_random_ranking": round(p_rand_r, 4),
    "rule_p_at_50": round(p_rule, 4),
    "n_labeled": int(len(labeled)),
    "n_frame_rows": int(len(frame)),
    "n_clients": int(labeled["client_hash_id"].nunique()),
    "decision_date": DECISION_DATE,
    "feature_window": f"{FEAT_LO}..{FEAT_HI}",
    "outcome_window": f"{OUT_LO}..{OUT_HI}",
    "model_type": "LogisticRegression(max_iter=500, C=1.0), no scaler, 5 features",
    "n_features": len(FEATURES),
    "split": "GroupKFold(5) by client_hash_id; random row split quoted for contrast only",
    "queue_score": "predicted_probability from a full-frame fit - in-sample, ranking quality only "
                   "ever quoted out-of-fold",
    "standardized_coefficients_grouped_fold": {
        "f_active_days": 0.6803, "f_ctr": -0.2782, "f_log_age_days": -0.2028,
        "f_avg_position": 0.1199, "f_log_impressions": 0.0526,
    },
    "permutation_importance_grouped_fold": {
        "f_ctr": 0.0602, "f_active_days": 0.0596, "f_log_age_days": 0.0399,
        "f_avg_position": 0.0010, "f_log_impressions": -0.0030,
    },
    "permutation_sd_grouped_fold": {
        "f_ctr": 0.0026, "f_active_days": 0.0015, "f_log_age_days": 0.0016,
        "f_avg_position": 0.0017, "f_log_impressions": 0.0010,
    },
    "permutation_importance_note": "f_ctr 0.0602 +/- 0.0026 and f_active_days 0.0596 +/- 0.0015 overlap "
                                  "within one SD, so the two are tied rather than ranked; f_avg_position "
                                  "0.0010 +/- 0.0017 is indistinguishable from zero",
    "reason_code_counts": {k: int(v) for k, v in q["reason_code"].value_counts().items()},
    "reason_code_decline_rate": {k: round(float(v), 4) for k, v in
                                 q.groupby("reason_code")["is_visibility_loss"].mean().items()},
    "unsupported_codes": [
        "stale_high_traffic (age quartiles non-monotonic, oldest declines least 0.3707; "
        "f_log_age_days coef -0.2028) - assigned to 0 pages",
        "low_activity_concentration as a DECLINE reason - f_active_days coef is +0.6803 (positive: more "
        "active days -> more decline); pages active <=10 days decline 0.1251 vs 0.4948 for 15/15 days, "
        "lift -0.3187. Kept as a monitor-only code; 79.2% of labeled pages sit at 15/15 active days, so a "
        "rank-based bottom-third cut fired on fully-active pages and was replaced by an absolute <=10 cut.",
    ],
    "no_refresh_action_assigned": "no pattern's measured lift over the base rate (best +0.0634) justifies "
                                 "telling an editor to change a page; every action in the queue is a look",
    "caveats": ["label is a within-month second-half vs first-half proxy, not future decline",
                "labeled population is pages with >=100 first-half impressions - high-traffic only",
                "38 clients, uneven (median 485 rows, max 17,610)",
                "grouped AUC 0.5902 is modest; model does not separate from the Week-4 rule at P@50",
                "no time-aware split exists; one decision date only",
                "decision-support triage order only - see the no-go list in section 3"],
    "source_notebook": "work/notebooks/w06_validation_audit.ipynb",
}
metrics_path = OUT_DIR / "playbook_metrics.json"
metrics_path.write_text(json.dumps(METRICS, indent=2) + "\n")
print(f"\nwrote {metrics_path.relative_to(ROOT)}")
W06 = {"base_rate": 0.4438, "grouped_auc": 0.5902, "grouped_p_at_50": 0.4800,
       "grouped_p_at_50_random_ranking": 0.4080, "random_auc": 0.6262, "random_p_at_50": 0.5600,
       "rule_p_at_50": 0.4360, "n_labeled": 77400, "n_clients": 38}
def _fmt(x):
    return f"{x:,}" if isinstance(x, int) else f"{x:.4f}"
receipt = pd.DataFrame([{"metric": k, "re-measured here": _fmt(METRICS[k]), "recorded in w06": _fmt(v),
                         "match": "yes" if abs(float(METRICS[k]) - float(v)) < 5e-4 else "NO"}
                        for k, v in W06.items()])
print("\n=== receipts: every playbook number re-measured in THIS notebook, out-of-fold ===")
print(receipt.to_string(index=False))
print("same estimator, same frame, same grouped split as w06 - so the playbook is built on numbers")
print("that reproduce here rather than on numbers copied forward and trusted.")

# ── the one figure: decline rate by score decile, base rate as the reference line ──
dec = (q.groupby("score_decile")
         .agg(n=("is_visibility_loss", "size"), decline_rate=("is_visibility_loss", "mean"))
         .reset_index())
fig, ax = plt.subplots(figsize=(7.2, 4.0))
ax.bar(dec["score_decile"], dec["decline_rate"], color="#4C6EF5", width=0.72, label="measured decline rate")
ax.axhline(BASE_RATE, color="#868E96", ls="--", lw=1.4, label=f"base rate {BASE_RATE:.4f}")
ax.set_xlabel("score decile (10 = highest predicted probability)")
ax.set_ylabel("is_visibility_loss rate")
ax.set_title("Queue head is where decline concentrates - in-sample, March 2026 slice", fontsize=11)
ax.text(0.02, 0.94, f"decile 1 {dec['decline_rate'].iloc[0]:.4f}  ->  decile 10 "
                    f"{dec['decline_rate'].iloc[-1]:.4f}   (n={len(q):,} labeled rows, 38 clients)",
        transform=ax.transAxes, fontsize=8.5, va="top", color="#495057")
ax.set_xticks(dec["score_decile"])
ax.legend(frameon=False, fontsize=9, loc="lower right")
for s in ["top", "right"]:
    ax.spines[s].set_visible(False)
ax.grid(axis="y", alpha=0.25)
fig.tight_layout()
fig_path = FIG_DIR / "queue_decline_profile.png"
fig.savefig(fig_path, dpi=150)
plt.close(fig)
print(f"\nwrote {fig_path.relative_to(ROOT)}")
print(f"decile 1 {dec['decline_rate'].iloc[0]:.4f} -> decile 10 {dec['decline_rate'].iloc[-1]:.4f} "
      f"(in-sample, so it is a consistency check on the ordering, NOT a performance claim;")
print(f" the out-of-fold figures to quote are grouped P@50 {p_grp_m:.4f} and grouped AUC {auc_grp:.4f}).")
print(dec.round(4).to_string(index=False))

# ── second figure: reason codes, how much queue each one holds and how well it does ──
# Two panels on purpose. Counts alone would make the biggest code look like the best one; the
# decline-rate panel next to the base rate is what shows which codes are actually earning a
# `review_for_refresh` and which are only a `monitor`.
fig2, (axL, axR) = plt.subplots(1, 2, figsize=(11.0, 4.2))

bc = by_code.reset_index()
cols = ["#4C6EF5" if v > BASE_RATE else "#ADB5BD" for v in bc["decline_rate"]]
axL.bar(bc["reason_code"], bc["share_of_queue"], color=cols, width=0.68)
for x, (share, n) in enumerate(zip(bc["share_of_queue"], bc["n"])):
    axL.text(x, share + 0.006, f"{share:.1%}", ha="center", fontsize=8.5, color="#495057")
axL.set_ylabel("share of the 77,400-row queue")
axL.set_title("How much of the queue each reason code holds", fontsize=10.5)
axL.set_xticks(np.arange(len(bc)))
axL.set_xticklabels(bc["reason_code"], rotation=32, ha="right", fontsize=8.5)
axL.set_ylim(0, max(bc["share_of_queue"]) * 1.18)

axR.bar(bc["reason_code"], bc["decline_rate"], color=cols, width=0.68)
axR.axhline(BASE_RATE, color="#868E96", ls="--", lw=1.4,
            label=f"base rate {BASE_RATE:.4f}")
for x, v in enumerate(bc["decline_rate"]):
    axR.text(x, v + 0.008, f"{v:.3f}", ha="center", fontsize=8.5, color="#495057")
axR.set_ylabel("measured is_visibility_loss rate")
axR.set_ylim(0, max(bc["decline_rate"]) * 1.20)
axR.set_title("Measured decline rate of the pages each code filed", fontsize=10.5)
axR.set_xticks(np.arange(len(bc)))
axR.set_xticklabels(bc["reason_code"], rotation=32, ha="right", fontsize=8.5)
axR.legend(frameon=False, fontsize=8.5, loc="upper right")

for ax in (axL, axR):
    for s in ["top", "right"]:
        ax.spines[s].set_visible(False)
    ax.grid(axis="y", alpha=0.22)
fig2.tight_layout()
fig2_path = FIG_DIR / "reason_code_breakdown.png"
fig2.savefig(fig2_path, dpi=150)
plt.close(fig2)
print(f"wrote {fig2_path.relative_to(ROOT)}")
grey = list(bc.loc[bc["decline_rate"] <= BASE_RATE, "reason_code"])
print(f"blue = assigned decline rate above the base rate {BASE_RATE:.4f}, grey = at or below it.")
print(f"grey bars: {grey}.")
print("The grey bars are the honest half of this figure, and one of them is a result rather than a")
print("defect: `low_activity_concentration` files the pages that decline LEAST, so it is filed, shown,")
print("and explicitly NOT turned into a refresh action.")


wrote work/outputs/action_playbook_queue.csv  rows 77,400 cols 13
columns: rank, client_hash_id, content_hash_id, predicted_probability, reason_code, action, f_log_impressions, f_ctr, f_avg_position, f_active_days, f_log_age_days, is_visibility_loss, score_decile
is_visibility_loss is carried for read-off only - it is the March label, not a future outcome.



wrote work/outputs/playbook_metrics.json

=== receipts: every playbook number re-measured in THIS notebook, out-of-fold ===
                        metric re-measured here recorded in w06 match
                     base_rate           0.4438          0.4438   yes
                   grouped_auc           0.5902          0.5902   yes
               grouped_p_at_50           0.4800          0.4800   yes
grouped_p_at_50_random_ranking           0.4080          0.4080   yes
                    random_auc           0.6262          0.6262   yes
                random_p_at_50           0.5600          0.5600   yes
                  rule_p_at_50           0.4360          0.4360   yes
                     n_labeled           77,400          77,400   yes
                     n_clients               38              38   yes
same estimator, same frame, same grouped split as w06 - so the playbook is built on numbers
that reproduce here rather than on numbers copied forward and trusted.

wrote work/

## Self-check

Confirming each line honestly:

- [x] **Two reason codes were corrected against the data, and both corrections are printed rather than buried.** `low_activity_concentration` was carrying the queue's only `review_for_refresh` action on a **bottom-third-by-rank** cut; the coefficient is **positive** (+0.6803) and 79.2% of labeled pages sit at 15/15 active days, so that cut was firing on *fully-active* pages and the group it filed declines **0.1251** against a 0.4438 base rate. The cut is now absolute (`<= 10` days), the action is `monitor_low_priority`, and §1.2 states the measured pattern in full. And **no code is assigned `review_for_refresh`** at all, because the best lift any single pattern shows is +0.0634.
- [x] **Every section above is filled** — markdown thinking *and* the code that backs it. §1's reason codes are cut from the labeled frame and their decline rates are printed next to them; §2's seven limits are six measured rows plus a count; §3's no-go list is printed and the false-positive risk in the queue's head is counted (healthy-looking pages in the top 200, and what they actually did); §4's triggers carry today's values and one measured noise band (the grouped fold spread).
- [x] **The notebook runs top to bottom with no errors** (executed with `jupyter nbconvert --execute --inplace`).
- [x] **No client names, URLs, or private queries anywhere.** Only pseudonymized hashes appear, and in the §1 preview they are truncated for printing. No HuggingFace token is in any cell: the warehouse is read from the local parquet cache and `scripts/` is never touched.
- [x] **My claims use careful words:** observed, measured, directional, decision-support, associated with. "Predicts", "causes" and "will decline" appear only inside the sentences that rule them out.
- [x] **Committed to my repo under `work/notebooks/`** — then submit my repo URL on the card. Done.

Four things I am **not** claiming, so nobody has to go looking for them:

1. **Not a win over the rule.** Grouped P@50 0.4800 vs the Week-4 rule's 0.4360 sits inside that rule's own 0.20–0.58 fold spread. Unseparated, not a win — and the rule was a comparator, never a feature.
2. **Not a decline predictor.** The label is second-half vs first-half impressions inside March 2026 on pages that already had ≥100 impressions. The score orders that population; it does not forecast.
3. **Not valid for low-traffic pages.** 74,581 of 151,981 frame rows have no label at all, and the labeled half is measurably more active than the unlabeled half. The queue is a high-traffic queue.
4. **Not a staleness detector.** `stale_high_traffic` is measured *against* in §1.4 — the oldest age quartile declines least (0.3707) and the pattern is not monotone — so it is assigned to zero pages rather than dressed up as a reason code.